In [1]:
from google.colab import drive; drive.mount('/content/drive')
import sys, importlib, numpy as np, pandas as pd, torch
from scipy.stats import wilcoxon
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate
BASE = "/content/drive/MyDrive/esci"

df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df

CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
def first_words(t, n): return " ".join(C._clean(t).split()[:n])
def short_plus(r, n):                     # short title + every other field
    parts = [first_words(r["product_title"], n)]
    b, c, f = C._clean(r["product_brand"]), C._clean(r["product_color"]), C._clean(r["product_bullet_point"])
    if b: parts.append(f"brand: {b}")
    if c: parts.append(f"color: {c}")
    if f: parts.append("features: " + f[:200])
    return " | ".join(parts)

for d in (train, val):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

rng = np.random.default_rng(7)                                       # fixed
train["text_aug_short_plus"] = [short_plus(r, int(rng.integers(3, 9))) for r in train.to_dict("records")]
vrecs = val.to_dict("records")
val["text_title_4words"] = [first_words(r["product_title"], 4) for r in vrecs]   # same as S9a/S9d
val["text_short4_plus"]  = [short_plus(r, 4) for r in vrecs]                     # same as S9d

TRAIN_STYLES = C.STYLES + ["aug_short_plus"]
EVAL_STYLES  = C.STYLES + ["title_4words", "short4_plus"]
print("train", len(train), "| val", len(val), "| GPU", torch.cuda.get_device_name(0))
print(train[["text_title", "text_aug_short_plus"]].head(3).to_string())

Mounted at /content/drive
train 65852 | val 11741 | GPU Tesla T4
                                                                                                                                                                                             text_title                                                                                                                                                                                                                                                                               text_aug_short_plus
0       Peiloh VR Headset Compatible with 4.7-6.3 inch iPhone and Android, Virtual Reality Headset with Wireless Headphones 3D VR Glasses Goggles for IMAX Movies & Cardboard Games, Soft & Comfortable        Peiloh VR Headset Compatible with 4.7-6.3 inch iPhone | brand: Peiloh | features: ✔Foldable and Lightweight Design: You can fold the VR Headset and the size after folding is only 6.3 * 4.7 inches, which will make it easy to carry and stor

In [2]:
LOG = "s9e_augplus_val_log.csv"
for seed in (42, 43, 44):
    cfg = {**T.CFG, "query_col": "q_focused_cat", "styles": TRAIN_STYLES, "eval_styles": EVAL_STYLES, "seed": seed}
    name = "minilm_l6_augplus__q_focused_cat" + ("" if seed == 42 else f"__s{seed}")
    print(f"\n===== {name} =====")
    T.train_run("cross-encoder/ms-marco-MiniLM-L-6-v2", name, train, val, evaluate, BASE, cfg=cfg, log_name=LOG)


===== minilm_l6_augplus__q_focused_cat =====
minilm_l6_augplus__q_focused_cat: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4917, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9217 0.9374 0.8454
 title_brand   0.9215 0.9363 0.8433
        full   0.9227 0.9377 0.8427
full_bullets   0.9262 0.9428 0.8421
   key_value   0.9238 0.9386 0.8439
title_4words   0.8779 0.8947 0.7942
 short4_plus   0.9008 0.9202 0.8229
  epoch 2 step 500/2058 loss 0.3614
  epoch 2 step 1000/2058 loss 0.3552
  epoch 2 step 1500/2058 loss 0.3515
  epoch 2 step 2000/2058 loss 0.3467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3465, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9253 0.9422 0.8470
 title_brand   0.9272 0.9425 0.8470
        full   0.9267 0.9403 0.8463
full_bullets   0.9273 0.9409 0.8442
   key_value   0.9273 0.9437 0.8454
title_4words   0.8771 0.8939 0.7954
 short4_plus   0.9000 0.9175 0.8204
  epoch 3 step 500/2058 loss 0.3192
  epoch 3 step 1000/2058 loss 0.3183
  epoch 3 step 1500/2058 loss 0.3185
  epoch 3 step 2000/2058 loss 0.3165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.3159, 1.7 min
       style  ndcg@10    mrr    p@5
       title   0.9263 0.9425 0.8470
 title_brand   0.9258 0.9402 0.8470
        full   0.9264 0.9406 0.8466
full_bullets   0.9278 0.9398 0.8451
   key_value   0.9277 0.9419 0.8463
title_4words   0.8781 0.8960 0.7963
 short4_plus   0.8995 0.9178 0.8213


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
def best(log_file, prefix):                    # best epoch on NORMAL formats only
    L = pd.read_csv(f"{BASE}/results/{log_file}"); L = L[L["style"].isin(C.STYLES)]
    L = L[L.run.str.fullmatch(rf"{prefix}__q_focused_cat(__s\d+)?")]
    ep = L.groupby(["run", "epoch"])["ndcg@10"].mean().reset_index()
    return ep.loc[ep.groupby("run")["ndcg@10"].idxmax()][["run", "epoch"]]
MODELS = {"mix": best("s7_grid_val_log.csv", "minilm_l6"), "augplus": best("s9e_augplus_val_log.csv", "minilm_l6_augplus")}
for g, b in MODELS.items(): print(g, b.values.tolist())

full = val["text_full"].values; short = val["text_title_4words"].values; splus = val["text_short4_plus"].values
val["is_short"] = np.random.default_rng(11).random(len(val)) < 0.5          # same half as S9d
SETUPS = {"all_full": full, "all_short": short, "all_short_plus": splus,
          "mixed_short": np.where(val.is_short, short, full),
          "mixed_short_plus": np.where(val.is_short, splus, full)}

def bias(v):
    v = v.copy(); v["pct"] = v.groupby("query_id")["score"].rank(ascending=False, pct=True)
    g = v[v.gain == 1].groupby(["query_id", "is_short"])["pct"].mean().unstack().dropna()
    return g[True] - g[False]

qtext = val["q_focused_cat"].tolist(); NDCG, BIAS, rows = {}, {}, []
for g, b in MODELS.items():
    for _, r in b.iterrows():
        path = f"{BASE}/models/{r.run}/epoch_{int(r.epoch)}"
        tok = AutoTokenizer.from_pretrained(path)
        m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
        for s, txt in SETUPS.items():
            v = val[["query_id", "gain", "is_short"]].copy()
            v["score"] = T.score_pairs(m, tok, qtext, list(txt), 256, 128)
            mtr, pq = evaluate(v); rows.append({"model": g, "run": r.run, "setup": s, **mtr})
            NDCG[(g, s)] = NDCG.get((g, s), []) + [pq.set_index("query_id")["ndcg@10"]]
            if s.startswith("mixed"): BIAS[(g, s)] = BIAS.get((g, s), []) + [bias(v)]
        del m; torch.cuda.empty_cache(); print("scored", r.run)

avg = lambda d, k: pd.concat(d[k], axis=1).mean(axis=1)
def p_of(x): x = x.dropna(); x = x[x != 0]; return wilcoxon(x).pvalue if len(x) else float("nan")
R = pd.DataFrame(rows)
print("\nNDCG@10 (mean of 3 seeds)\n", R.groupby(["setup", "model"])["ndcg@10"].mean().unstack().reindex(list(SETUPS)).round(4).to_string())
print("\nBias (Exact short minus full, list percentile):")
for g in MODELS:
    for s in ["mixed_short", "mixed_short_plus"]:
        print(f"  {g:8s} {s:16s}: {avg(BIAS, (g, s)).mean():+.4f}")

db = (avg(BIAS, ("augplus", "mixed_short_plus")) - avg(BIAS, ("mix", "mixed_short_plus"))).dropna()
dn = avg(NDCG, ("augplus", "all_full")) - avg(NDCG, ("mix", "all_full"))
p_b, p_n = p_of(db), p_of(dn)
print(f"\n(1) bias augplus - mix [mixed_short_plus]: {db.mean():+.4f} | p = {p_b:.4f}")
print(f"(2) all_full NDCG augplus - mix          : {dn.mean():+.4f} | p = {p_n:.4f}")
ok1 = db.mean() < 0 and p_b < 0.05
ok2 = dn.mean() >= 0 or p_n >= 0.05
print(f"bias significantly smaller: {ok1} | all_full not significantly worse: {ok2}")
print("DECISION:", "use AUGPLUS model as final" if (ok1 and ok2) else "keep CURRENT (mix) model")
R.to_csv(f"{BASE}/results/s9e_augplus_vs_mix.csv", index=False)

mix [['minilm_l6__q_focused_cat', 3], ['minilm_l6__q_focused_cat__s43', 3], ['minilm_l6__q_focused_cat__s44', 2]]
augplus [['minilm_l6_augplus__q_focused_cat', 3], ['minilm_l6_augplus__q_focused_cat__s43', 2], ['minilm_l6_augplus__q_focused_cat__s44', 3]]


Loading weights:   0%|          | 0/105 [00:03<?, ?it/s]

scored minilm_l6__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6__q_focused_cat__s43


...

scored minilm_l6_augplus__q_focused_cat


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_augplus__q_focused_cat__s43


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

scored minilm_l6_augplus__q_focused_cat__s44

NDCG@10 (mean of 3 seeds)
 model             augplus     mix
setup                            
all_full           0.9269  0.9278
all_short          0.8798  0.8811
all_short_plus     0.9007  0.8998
mixed_short        0.8976  0.8966
mixed_short_plus   0.9077  0.9073

Bias (Exact short minus full, list percentile):
  mix      mixed_short     : +0.2679
  mix      mixed_short_plus: +0.1842
  augplus  mixed_short     : +0.2528
  augplus  mixed_short_plus: +0.0395

(1) bias augplus - mix [mixed_short_plus]: -0.1446 | p = 0.0000
(2) all_full NDCG augplus - mix          : -0.0009 | p = 0.3852
bias significantly smaller: True | all_full not significantly worse: True
DECISION: use AUGPLUS model as final
